### **Imports Iniciales**

In [3]:
import re, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# NLP clásico
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# Representación y modelo
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import NearestNeighbors

# Remuestreo (para la evaluación de SMOTE)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE

warnings.filterwarnings(action='ignore')
nltk.download('stopwords', quiet=True)   # la tokenización se hace con regex: no se necesita 'punkt'
plt.style.use(style='seaborn-v0_8-whitegrid')

SEMILLA = 42
CLASES = ['negativo', 'neutral', 'positivo']
print('✓ Entorno listo')

✓ Entorno listo


Parte 1: carga de datos

In [4]:
train_raw = pd.read_csv('data/train.csv')
eval_raw = pd.read_csv('data/eval.csv')

display(train_raw.head())

,id,text,label
0,0,Lo pedí un martes por la noche y me llegó al d...,neutral
1,1,Se lo compré a un familiar después de pensarla...,positivo
2,2,Lo pedí a mediados de mes y me llegó en tres d...,negativo
3,3,Compré esto en línea y llegó en unos tres días...,negativo
4,4,Compré este kit el mes pasado porque necesitab...,positivo


Parte 2: Calidad de datos

Para la verificación de la calidad de datos, se parte por la siguiente hipótesis: Que la columna *id* sea única no garantiza que los datos no lo sean, por lo que se va a verificar la **dimensiones de calidad** de las filas en dos niveles:

1. **Estructural:** *id* único, nulos, textos vacíos, etiquetas válidas. Esto quiere decir que en este nivel, se verificará la **unicidad**, **completitud** y **consistencia** en su forma "clásica".
2. **De contenido:** Textos repetidos (exactos y casi exactos) y, sobre todo, textos con **etiquetas distintas** (ruido de etiqueta). Es decir, se verificará la **unicidad** y **consistencia** de los datos.

Se hace esta verificación porque si una reseña duplicada cae en el fold de entrenamiento y en el de validación a la vez, la exactitud de validación se infla (data leakage) y elegiríamos hiperparámetros con una métrica optimista (sobreestima los datos).

In [6]:
def auditar(df, nombre, con_label=True):
    print(f'-- {nombre} --')
    print(f'Filas: {len(df):,}')
    print(f'Nulos por columna: {df.isna().sum().to_dict()}')
    print(f'IDs únicos: {df["id"].is_unique} (IDs repetidos: {df["id"].duplicated().sum()})')
    print(f'Textos vacíos o solo espacios: {df['text'].fillna("").str.strip().eq("").sum()}')
    if con_label:
        etiquetas = df["label"].astype(str).str.strip().str.lower()
        fuera = sorted(set(etiquetas) - set(CLASES))
        print(f'Etiquetas fuera de {CLASES}: {fuera if fuera else "ninguna"}')
        print(f'Etiquetas con espacios/Mayúsculas por corregir: {(etiquetas != df["label"]).sum()}')
    print()

auditar(train_raw, 'train')
auditar(eval_raw, 'eval', con_label = False)

-- train --
Filas: 12,000
Nulos por columna: {'id': 0, 'text': 0, 'label': 0}
IDs únicos: True (IDs repetidos: 0)
Textos vacíos o solo espacios: 0
Etiquetas fuera de ['negativo', 'neutral', 'positivo']: ninguna
Etiquetas con espacios/Mayúsculas por corregir: 0

-- eval --
Filas: 3,000
Nulos por columna: {'id': 0, 'text': 0}
IDs únicos: True (IDs repetidos: 0)
Textos vacíos o solo espacios: 0



In [7]:
#Se crea una columna auxiliar 'clave' "laxa" para comparar el parecido con 'text'

def clave_duplicado(s):
    #Estandariza espacios
    return s.fillna('').str.lower().str.replace(r'\s+', ' ', regex=True)

#Se hace una copia del dataset de entrenamiento como buena práctica
train = train_raw.copy()
train['label'] = train['label'].astype(str).str.strip().str.lower()
train['clave'] = clave_duplicado(train['text'])

grupos = train.groupby('clave')['label'].agg(['size', 'nunique'])
dup_mismo_label = grupos[(grupos['size'] > 1) & (grupos['nunique'] == 1)]
dup_conflicto = grupos[grupos['nunique'] > 1]

print(f'Duplicados exactos (texto crudo): {train["text"].duplicated().sum()}')
print(f'Duplicados por clave laxa: {train["clave"].duplicated().sum()}')
print(f'grupos con la misma etiqueta: {len(dup_mismo_label)} (filas sobrantes: {(dup_mismo_label["size"] - 1).sum()})')
print(f'Duplicados exactos (texto crudo): {len(dup_conflicto)} (filas involucradas: {dup_conflicto["size"].sum()})')

if len(dup_conflicto):
    display(train[train['clave'].isin(dup_conflicto.index)].sort_values('clave')[['id', 'text', 'label']].head(10))

#Unicamente se verifica si hay datos solapados entre datos de train y eval
solapamiento = set(train['clave']) & set(clave_duplicado(eval_raw['text']))
print(f'Textos de eval que también aparecen en train: {len(solapamiento)}')

Duplicados exactos (texto crudo): 0
Duplicados por clave laxa: 0
grupos con la misma etiqueta: 0 (filas sobrantes: 0)
Duplicados exactos (texto crudo): 0 (filas involucradas: 0)
Textos de eval que también aparecen en train: 0


In [8]:
n_inicial = len(train)
train = train[train['clave'] != '']
train = train[~train['clave'].isin(dup_conflicto.index)]
train = train.drop_duplicates(subset='clave', keep = 'first')
train = train.drop(columns='clave').reset_index(drop=True)

eval_df = eval_raw.copy()
eval_df['text'] = eval_df['text'].fillna('')

print(f'train: {n_inicial:,}: {len(train):,} filas (eliminadas: {n_inicial - len(train):,})')

train: 12,000: 12,000 filas (eliminadas: 0)


3. Exploracion rapida

- balanceo de clases
- longitud de reseñas
- Diferencia de palabras de negacion y contraste por clase (para no tratarlas como stop words)